# 🧬 Classical ML Baselines for SP-Family TF Binding Prediction

This notebook trains 10 classical machine learning models on the processed dataset of transcription factor binding sites for the SP family (SP1, SP2, SP4) and Negative controls. 

It supports:
1. **One-hot encoding** (flattened to 404 features).
2. **k-mer TF-IDF representation** (extracts counts of all DNA subsequences of length k).

In [ ]:
# Cell 1: Clone repository if running in Google Colab (uncomment if needed)
# !git clone https://github.com/JustinYuanZe/SP1_TF_Biding_Project.git
# %cd SP1_TF_Biding_Project
# !pip install scikit-learn pandas numpy matplotlib

In [ ]:
# Cell 2: Imports and DNA feature extraction utilities
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import BernoulliNB
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

def load_fasta_seqs(fasta_path):
    """Load sequences from a FASTA file."""
    sequences = []
    with open(fasta_path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line.startswith('>'):
                sequences.append(line.upper())
    return sequences

def seqs_to_onehot(sequences):
    """Convert DNA sequences to flattened one-hot representation."""
    n_samples = len(sequences)
    seq_array = np.array([list(seq) for seq in sequences], dtype='U1')
    onehot = np.zeros((n_samples, 101, 4), dtype=np.int8)
    onehot[seq_array == 'A', 0] = 1
    onehot[seq_array == 'C', 1] = 1
    onehot[seq_array == 'G', 2] = 1
    onehot[seq_array == 'T', 3] = 1
    return onehot.reshape(n_samples, -1)

def extract_kmer_features(sequences, ngram_range=(4, 6), vectorizer_type='tfidf'):
    """Extract k-mer features using CountVectorizer or TfidfVectorizer."""
    if vectorizer_type == 'tfidf':
        vectorizer = TfidfVectorizer(analyzer='char', ngram_range=ngram_range)
    else:
        vectorizer = CountVectorizer(analyzer='char', ngram_range=ngram_range)
    return vectorizer.fit_transform(sequences)

In [ ]:
# Cell 3: Load balanced datasets from data/processed/
sp1_path = os.path.join("data", "processed", "sp1_positive_final.fasta")
sp2_path = os.path.join("data", "processed", "sp2_positive_final.fasta")
sp4_path = os.path.join("data", "processed", "sp4_positive_final.fasta")
neg_path = os.path.join("data", "processed", "negative_final.fasta")

print("Loading sequences...")
seqs_sp1 = load_fasta_seqs(sp1_path)
seqs_sp2 = load_fasta_seqs(sp2_path)
seqs_sp4 = load_fasta_seqs(sp4_path)
seqs_neg = load_fasta_seqs(neg_path)

print(f"SP1 Positive: {len(seqs_sp1)} samples")
print(f"SP2 Positive: {len(seqs_sp2)} samples")
print(f"SP4 Positive: {len(seqs_sp4)} samples")
print(f"Negative (Hard Control): {len(seqs_neg)} samples")

In [ ]:
# Cell 4: Select Task and Extract Features
# Select task: 'binary' (SP1 vs Neg) or '4class' (SP1, SP2, SP4, Neg)
TASK = 'binary'  

# Select representation: 'onehot' or 'tfidf' or 'count'
REPRESENTATION = 'tfidf'
K_MIN, K_MAX = 4, 6

if TASK == 'binary':
    sequences = seqs_sp1 + seqs_neg
    y = np.concatenate([np.ones(len(seqs_sp1)), np.zeros(len(seqs_neg))], axis=0)
else:
    sequences = seqs_sp1 + seqs_sp2 + seqs_sp4 + seqs_neg
    y = np.concatenate([
        np.zeros(len(seqs_sp1)),
        np.ones(len(seqs_sp2)),
        np.full(len(seqs_sp4), 2),
        np.full(len(seqs_neg), 3)
    ], axis=0)

print(f"Extracting features: {REPRESENTATION}...")
if REPRESENTATION == 'onehot':
    X = seqs_to_onehot(sequences)
else:
    X = extract_kmer_features(sequences, ngram_range=(K_MIN, K_MAX), vectorizer_type=REPRESENTATION)

print(f"Feature matrix shape: {X.shape}")

In [ ]:
# Cell 5: Train and evaluate all 10 baseline classifiers
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}\n")

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1),
    "Decision Tree": DecisionTreeClassifier(max_depth=10, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=20, n_jobs=-1, random_state=42),
    "Extra Trees": ExtraTreesClassifier(n_estimators=100, max_depth=20, n_jobs=-1, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, max_depth=5, random_state=42),
    "AdaBoost": AdaBoostClassifier(n_estimators=100, random_state=42),
    "SVM (Linear)": SVC(kernel='linear', random_state=42),
    "KNN (k=5)": KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    "Naive Bayes": BernoulliNB(),
    "MLP": MLPClassifier(hidden_layer_sizes=(100,), max_iter=500, random_state=42),
}

results = []
print("Training 10 models...")
for name, model in models.items():
    import warnings
    warnings.filterwarnings("ignore", category=FutureWarning)
    
    start_time = time.time()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    elapsed = time.time() - start_time
    print(f"{name:<20} | Accuracy: {acc:.4f} | Time: {elapsed:.1f}s")
    
    results.append({
        "Model": name,
        "Accuracy": acc,
        "Time (s)": elapsed
    })

In [ ]:
# Cell 6: Visualize baseline performance comparison
df_results = pd.DataFrame(results).sort_values(by="Accuracy", ascending=False)

plt.figure(figsize=(12, 6))
plt.bar(df_results["Model"], df_results["Accuracy"], color='#1f77b4', alpha=0.85)
plt.axhline(y=0.5 if TASK == 'binary' else 0.25, color='r', linestyle='--', label='Random Baseline')
plt.title(f"Classical ML Accuracy Comparison ({TASK.upper()} Task)", fontsize=14, pad=15)
plt.ylabel("Accuracy", fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.ylim(0, 1.0)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()